# Solutions — Deduplication: Exact, Latest-Wins, Fuzzy, Survivorship

Exercise notebook: `08_data_engineering_python/04_deduplication.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
DATA = LAB / "datasets"
ORDERS_CSV = DATA / "csv" / "orders.csv"          # 5k+ messy orders
CUSTOMERS_CSV = DATA / "csv" / "customers.csv"
PRODUCTS_CSV = DATA / "csv" / "products.csv"
OUTPUT = LAB / "output" / "08"                       # scratch output folder (git-ignored)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Exercise 1 — 🟢 Easy · Exact duplicate rows

**Working solution**

In [ ]:
def dedupe_exact(rows):
    seen, unique = set(), []
    for row in rows:
        fingerprint = frozenset(row.items())
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        unique.append(row)
    return unique, len(rows) - len(unique)

**Explanation**

Dicts aren't hashable, but `frozenset(row.items())` is — and ignores key order. (For rows with unhashable values, hash a canonical `json.dumps(row, sort_keys=True)`.)

**Complexity:** O(n × fields).

**Common mistakes**

- Comparing every row with every other row.

## Exercise 2 — 🟡 Medium · Latest version per business key

**Working solution**

In [ ]:
def dedupe_latest(rows, key, order_by):
    fields = (key,) if isinstance(key, str) else tuple(key)
    best = {}
    for row in rows:
        k = tuple(row[f] for f in fields)
        if k not in best or row[order_by] >= best[k][order_by]:
            best[k] = row
    return list(best.values())

**Explanation**

Dict assignment to an existing key keeps the key's original position — so first-appearance order is preserved while the value is replaced by newer versions.

**Complexity:** O(n).

**Common mistakes**

- Sorting the whole dataset first when a single pass suffices.

## Exercise 3 — 🟡 Medium · Duplicate report

**Working solution**

In [ ]:
from collections import Counter

def duplicate_report(rows, key):
    counts = Counter(r[key] for r in rows)
    dupes = {k: c for k, c in counts.items() if c > 1}
    return {"rows": sum(counts.values()), "distinct_keys": len(counts), "duplicate_keys": len(dupes),
            "duplicate_rows": sum(c - 1 for c in dupes.values()),
            "worst": sorted(dupes.items(), key=lambda kc: (-kc[1], kc[0]))[:5]}

**Explanation**

Profile before you dedupe: knowing *how many* duplicates and their distribution tells you whether it's a retry storm (a few keys ×100) or systemic (many keys ×2).

**Complexity:** O(n + d log d).

**Common mistakes**

- Reporting `rows - distinct` as 'duplicate keys' (that's duplicate rows).

## Exercise 4 — 🟠 Hard · Fuzzy duplicates with blocking

**Working solution**

In [ ]:
import unicodedata
from collections import defaultdict
from difflib import SequenceMatcher
from itertools import combinations

def normalize_name(name):
    decomposed = unicodedata.normalize("NFKD", name or "")
    letters = "".join(ch if ch.isalpha() or ch.isspace() else " " for ch in decomposed if not unicodedata.combining(ch))
    return " ".join(letters.lower().split())

def fuzzy_duplicates(customers, threshold=0.9):
    blocks = defaultdict(list)
    for c in customers:
        norm = normalize_name(c.get("name"))
        if norm:
            blocks[(c.get("country"), norm[0])].append((c["customer_id"], norm))
    pairs = []
    for members in blocks.values():
        for (id_a, a), (id_b, b) in combinations(members, 2):
            ratio = SequenceMatcher(None, a, b).ratio()
            if ratio >= threshold:
                lo, hi = sorted((id_a, id_b))
                pairs.append((lo, hi, round(ratio, 3)))
    return sorted(pairs, key=lambda p: (-p[2], p[0], p[1]))

**Explanation**

**Blocking** is the key idea in entity resolution: compare only records that could plausibly match, turning n² into Σ(block²). `itertools.combinations` yields each pair once.
Unicode normalization makes `Müller` ≈ `Muller`.

**Complexity:** O(Σ block_size² × name_len²).

**Common mistakes**

- Comparing all pairs of 100k customers (5 billion comparisons).
- Comparing raw names so case and punctuation dominate the score.

## Challenge — 🔴 Very Hard · Golden records with survivorship rules

**Working solution**

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime

_SEGMENT_RANK = {"retail": 0, "smb": 1, "enterprise": 2}

def _date(text):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime((text or "").strip(), fmt).date()
        except ValueError:
            pass
    return None

def build_golden_records(customers):
    groups = defaultdict(list)
    for c in customers:
        email = (c.get("email") or "").strip().lower()
        groups[("email", email) if email else ("id", c["customer_id"])].append(c)
    golden = []
    for (kind, value), members in groups.items():
        dated = [(m, _date(m.get("signup_date"))) for m in members]
        names = [(" ".join((m.get("name") or "").split()).title(), d) for m, d in dated]
        names = [(n, d) for n, d in names if n]
        best_name = max(names, key=lambda nd: (len(nd[0]), nd[1] or datetime.min.date()))[0] if names else None
        countries = Counter((m.get("country") or "").strip().upper() for m in members if (m.get("country") or "").strip())
        country = min(countries.items(), key=lambda kv: (-kv[1], kv[0]))[0] if countries else None
        dates = [d for _, d in dated if d]
        segments = [m.get("segment") for m in members if m.get("segment") in _SEGMENT_RANK]
        golden.append({
            "golden_id": min(m["customer_id"] for m in members),
            "source_ids": sorted(m["customer_id"] for m in members),
            "name": best_name,
            "email": value if kind == "email" else None,
            "country": country,
            "signup_date": min(dates).isoformat() if dates else None,
            "is_active": any(str(m.get("is_active", "")).lower() == "true" for m in members),
            "segment": max(segments, key=_SEGMENT_RANK.get) if segments else None,
        })
    return sorted(golden, key=lambda g: g["golden_id"])

**Explanation**

Survivorship = explicit, per-attribute rules for which source value 'wins'. Writing each rule as a tiny `max`/`min`/`Counter` expression keeps them reviewable by the business.
This is MDM (master data management) in miniature.

**Complexity:** O(n log n) overall.

**Common mistakes**

- Picking 'the first row' for every attribute — arbitrary and non-deterministic across runs.
- Grouping rows without email together under `''`.